## Dependencias

In [38]:
import pandas as pd
import plotly.express as px
import numpy as np

from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb

pd.options.display.float_format = '{:.4f}'.format
pd.set_option('display.max_columns', 200)

## Lectura de Datos

In [2]:
df = pd.read_parquet(r"D:\Diplomado\TecnicasCognitivasIntroduccionABD\Proyecto\DataCleaned\df_full.parquet")

df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,master_id,sell_price,master_id_1
0,HOBBIES_1_169_WI_3_validation,HOBBIES_1_169,HOBBIES_1,HOBBIES,WI_3,WI,d_1624,0,2015-07-10,11523,Friday,7,7,2015,d_1624,None,None,None,None,1,0,0,HOBBIES_1_169_WI_3_11523,0.57,HOBBIES_1_169_WI_3_11523
1,HOBBIES_1_171_WI_3_validation,HOBBIES_1_171,HOBBIES_1,HOBBIES,WI_3,WI,d_1624,0,2015-07-10,11523,Friday,7,7,2015,d_1624,None,None,None,None,1,0,0,HOBBIES_1_171_WI_3_11523,2.88,HOBBIES_1_171_WI_3_11523
2,HOBBIES_1_172_WI_3_validation,HOBBIES_1_172,HOBBIES_1,HOBBIES,WI_3,WI,d_1624,0,2015-07-10,11523,Friday,7,7,2015,d_1624,None,None,None,None,1,0,0,HOBBIES_1_172_WI_3_11523,2.98,HOBBIES_1_172_WI_3_11523
3,HOBBIES_1_173_WI_3_validation,HOBBIES_1_173,HOBBIES_1,HOBBIES,WI_3,WI,d_1624,2,2015-07-10,11523,Friday,7,7,2015,d_1624,None,None,None,None,1,0,0,HOBBIES_1_173_WI_3_11523,10.97,HOBBIES_1_173_WI_3_11523
4,HOBBIES_1_174_WI_3_validation,HOBBIES_1_174,HOBBIES_1,HOBBIES,WI_3,WI,d_1624,0,2015-07-10,11523,Friday,7,7,2015,d_1624,None,None,None,None,1,0,0,HOBBIES_1_174_WI_3_11523,4.88,HOBBIES_1_174_WI_3_11523


In [3]:
df = df.sort_values(["id", "date"]).reset_index(drop=True).copy()

In [4]:
df_ = df[['id',
          'item_id',
          'dept_id',
          'cat_id',
          'store_id',
          'state_id',
          'date',
          'wm_yr_wk',
          'wday',
          'month',
          'year',
          'event_type_1',
          'snap_CA',
          'snap_TX',
          'snap_WI',
          'sales',
          'sell_price']].copy()

## Feature Engineering

In [ ]:
# Colisionamos las variables snap

df_['snap_active'] = 0

df_.loc[df_['state_id'] == 'CA', 'snap_active'] = df_['snap_CA']
df_.loc[df_['state_id'] == 'TX', 'snap_active'] = df_['snap_TX']
df_.loc[df_['state_id'] == 'WI', 'snap_active'] = df_['snap_WI']

df_.drop(columns=['snap_CA', 'snap_TX', 'snap_WI'], inplace=True)

In [6]:
# Revisamos vacíos 

df_.isna().sum()/df_.shape[0]

id              0.000000
item_id         0.000000
dept_id         0.000000
cat_id          0.000000
store_id        0.000000
state_id        0.000000
date            0.000000
wm_yr_wk        0.000000
wday            0.000000
month           0.000000
year            0.000000
event_type_1    0.923077
sales           0.000000
sell_price      0.018855
snap_active     0.000000
dtype: float64

Notamos `event_type_1` tiene el 92% de vacíos, pero esto son solo porque no todos los días son días festivos. En este caso el vacío es una categoría. Por el otro lado, `sell_price` tiene el 1.9% de vacíos. Estos son productos que no habían o todavía no se integreaban. Estos se eliminarán.

In [7]:
df_.dropna(subset=['sell_price'], inplace=True)#.isna().sum()/df_.shape[0]

In [8]:
df_fe = df_.copy()

In [9]:
# LAGS

group = df_fe.groupby('id')

lags = [28, 35, 42, 56]

for lag in lags:
    df_fe[f'lag_{lag}'] = group['sales'].shift(lag)

In [ ]:
# Rolling

windows = [28, 35, 42]

for w in windows:
    df_fe[f'rmean_{w}'] = group['sales'].shift(1).rolling(w, min_periods=1).mean()
    df_fe[f'rstd_{w}'] = group['sales'].shift(1).rolling(w, min_periods=1).std()
    df_fe[f'rmax_{w}'] = group['sales'].shift(1).rolling(w, min_periods=1).max()
    df_fe[f'rmin_{w}'] = group['sales'].shift(1).rolling(w, min_periods=1).min()

In [11]:
df_fe['lag_35_rolling_28'] = group['sales'].shift(35).rolling(7).mean()
df_fe['lag_28_rolling_28'] = group['sales'].shift(28).rolling(7).mean()

In [12]:
# Diferencias

df_fe['diff_28_35'] = df_fe['lag_28'] - df_fe['lag_35']
df_fe['diff_35_42'] = df_fe['lag_35'] - df_fe['lag_42']

In [13]:
# Precios

df_fe['price_lag_28'] = group['sell_price'].shift(28)

df_fe['price_chg_28'] = group['sell_price'].pct_change(28)
df_fe['price_max_year'] = group['sell_price'].transform(lambda x: x.rolling(365, min_periods=1).max())
df_fe['norm_price'] = df_fe['sell_price'] / df_fe['price_max_year']
df_fe['price_std_28'] = group['sell_price'].transform(lambda x: x.rolling(28).std())

In [14]:
# SNAP

df_fe['event_type_1'] = df_fe['event_type_1'].fillna('None')
df_fe['event_type_1'].value_counts()

event_type_1
None         18663010
National       568259
Religious      538736
Cultural       331628
Sporting       120980
Name: count, dtype: int64

In [15]:
df_fe['is_event'] = (df_fe['event_type_1']!='None').astype(int)

df_fe['snap_event'] = df_fe['snap_active'] * df_fe['is_event']

In [16]:
# agregaciones

df_fe['store_sales_mean_lag_28'] = df_fe.groupby(['store_id'])['sales'].transform(lambda x: x.shift(28).rolling(7).mean())
df_fe['cat_sales_mean_lag_28'] = df_fe.groupby(['cat_id'])['sales'].transform(lambda x: x.shift(28).rolling(7).mean())
df_fe['dept_sales_mean_lag_28'] = df_fe.groupby(['dept_id'])['sales'].transform(lambda x: x.shift(28).rolling(7).mean())

In [17]:
# Eliminamos wm_yr_wk porque tiene índice de tiempo implícito
df_fe.drop(columns=['wm_yr_wk'], inplace=True)

In [18]:
df_fe['wday_sin'] = np.sin(2 * np.pi*df_fe['wday']/7)
df_fe['wday_cos'] = np.cos(2*np.pi*df_fe['wday']/7)

In [19]:
df_fe['month_sin'] = np.sin(2 * np.pi*df_fe['month']/12)
df_fe['month_cos'] = np.cos(2*np.pi*df_fe['month']/12)

In [20]:
df_fe['week'] = pd.to_datetime(df_fe['date']).dt.isocalendar().week.astype(int)

df_fe["week_sin"] = np.sin(2 * np.pi * df_fe["week"] / 52)
df_fe["week_cos"] = np.cos(2 * np.pi * df_fe["week"] / 52)

## Definición de Variables

In [21]:
for h in range(1, 28 +1):
    df_fe[f'target_{h}'] = df_fe.groupby('id')['sales'].shift(-h)

In [22]:
# Target multi-out

tgt = [f'target_{h}' for h in range(1, 29)]

In [29]:
# Unidad Muestral
um = ['id']

# Variables discretas
vard = ['item_id', 'dept_id', 'cat_id', 'event_type_1']

# Variables Continuas
varc = ['lag_28', 'lag_35', 'lag_42', 'lag_56', 'rmean_28',
       'rstd_28', 'rmax_28', 'rmin_28', 'rmean_35', 'rstd_35', 'rmax_35',
       'rmin_35', 'rmean_42', 'rstd_42', 'rmax_42', 'rmin_42',
       'lag_35_rolling_28', 'lag_28_rolling_28', 'diff_28_35', 'diff_35_42',
       'price_lag_28', 'price_chg_28', 'price_max_year', 'norm_price',
       'price_std_28', 'store_sales_mean_lag_28',
       'cat_sales_mean_lag_28', 'dept_sales_mean_lag_28', 'wday_sin',
       'wday_cos', 'month_sin', 'month_cos', 'week', 'week_sin', 'week_cos']

# Variables Discretas
snaps = ['snap_active','snap_event']

feat = vard + varc 

### Variables Discretas

In [30]:
# Haremos Dynamic Encoding para respetar el tiempo

def dynamic_target_encoding(df, col, target="sales"):
    return df.sort_values('date').groupby(col)[target].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    

for col in vard:
    df_fe[f"d_{col}"] = dynamic_target_encoding(df_fe, col)
    

In [ ]:
vard = [c for c in df_fe.columns if c.startswith('d_')]

In [ ]:
aux = (df_fe.isna().sum()/df_fe.shape[0]).reset_index(name='nans')

aux = aux.sort_values('nans', ascending=False).reset_index(drop=True)

aux.head(10)

,index,nans
0,lag_56,0.084432
1,lag_42,0.063324
2,rmean_42,0.063324
3,rstd_42,0.063324
4,rmax_42,0.063324
5,rmin_42,0.063324
6,diff_35_42,0.063324
7,lag_35_rolling_28,0.061816
8,rstd_35,0.052770
9,rmean_35,0.052770


Como notamos, hay muchos valores vacíos resultado de los lags. Para tratar con esto y no tener una pérdida masiva de información primero nos quedaremos con los valores válidos del target.

In [25]:
df_fe.dropna(subset=tgt, inplace=True, axis=0)

In [26]:
aux = (df_fe.isna().sum()/df_fe.shape[0]).reset_index(name='nans')

aux = aux.sort_values('nans', ascending=False)

aux.loc[aux['nans']!=0].head(10)

,index,nans
17,lag_56,0.088149
28,rmax_42,0.066115
33,diff_35_42,0.066115
26,rmean_42,0.066115
29,rmin_42,0.066115
16,lag_42,0.066115
27,rstd_42,0.066115
30,lag_35_rolling_28,0.064541
23,rstd_35,0.055096
15,lag_35,0.055096


In [40]:
df_fe.dropna(subset=[c for c in df_fe.columns if c.startswith('lag_')], inplace=True)

In [42]:
auxii = (df_fe.isna().sum() / df_fe.shape[0]).reset_index(name='nans')

auxii = auxii.sort_values('nans',ascending=False).reset_index(drop=True)

In [43]:
auxii.head(10)

,index,nans
0,d_event_type_1,0.0000
1,item_id,0.0000
2,id,0.0000
3,cat_id,0.0000
4,store_id,0.0000
5,state_id,0.0000
6,date,0.0000
7,wday,0.0000
8,month,0.0000
9,year,0.0000


In [45]:
df_fe.to_parquet(r"D:\Diplomado\TecnicasCognitivasIntroduccionABD\Proyecto\DataCleaned\df_fe.parquet")